# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- 1000건마다 `1000.parquet`, `2000.parquet`, ... 형태로 순차 저장
- 중단 후 이어서 크롤링 가능 (기존 파일 자동 감지)
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드

In [ ]:
import os
import time
import random
import requests
import queue
import threading
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
from fake_useragent import UserAgent

# --- 사용자 설정 ---
END_ID = 1               # 수집을 종료할 최소 ID
WORKERS_PER_SITE = 3     # 사이트당 스레드 수 (총 9개)
CHUNK_SIZE = 1000        # 저장 단위
SAVE_DIR = r"../data/parquet"

DELAY_MIN = 0.1
DELAY_MAX = 0.2
BLOCK_SLEEP = 60        # 차단 시 대기 시간 (초)

# 대상 사이트 API 정보
SITES = {
    's': "https://safebooru.org/index.php",
    'g': "https://gelbooru.com/index.php",
    'd': "https://danbooru.donmai.us/posts.xml"
}

# --- 전역 변수 및 동기화 객체 ---
buffer = []
next_chunk_num = 1
data_lock = threading.Lock()

# 사이트별 독립적인 큐와 정지(Pause) 이벤트
queues = {site: queue.Queue(maxsize=1000) for site in SITES.keys()}
pause_events = {site: threading.Event() for site in SITES.keys()}
for event in pause_events.values():
    event.set() # 기본 상태는 '진행'

ua = UserAgent()
session = requests.Session()
stats = {site: {'success': 0, 'empty': 0, 'denied': 0, 'error': 0} for site in SITES.keys()}
saved_chunks = 0

# ───────────────────────────────────────────────
# API에서 가장 최신(Max) ID 가져오기
# ───────────────────────────────────────────────
def get_latest_id_from_api(site):
    url = SITES[site]
    params = {'limit': 1}
    if site in ['s', 'g']:
        params.update({'page': 'dapi', 's': 'post', 'q': 'index'})
    
    try:
        response = session.get(url, params=params, headers={'User-Agent': ua.random}, timeout=10)
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            post = root.find('post')
            if post is not None:
                latest_id = int(post.get('id'))
                print(f"[{site}] API 최신 ID 확인: {latest_id:,}")
                return latest_id
    except Exception as e:
        print(f"[{site}] 최신 ID 조회 실패: {e}")
    
    # 실패 시 임의의 큰 숫자 반환 (안전 장치)
    return 10000000

# ───────────────────────────────────────────────
# 파켓 파일 분석 및 사이트별 시작 ID 계산
# ───────────────────────────────────────────────
def get_resume_info():
    global next_chunk_num
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)
        
    files = [f for f in os.listdir(SAVE_DIR) if f.endswith('.parquet') and f.replace('.parquet', '').isdigit()]
    files = sorted(files, key=lambda x: int(x.replace('.parquet', '')))
    
    # 각 사이트별 수집된 가장 작은 ID를 저장할 딕셔너리
    min_ids = {'s': float('inf'), 'g': float('inf'), 'd': float('inf')}
    
    if files:
        print("기존 파켓 파일들을 스캔하여 사이트별 최소 ID를 찾습니다...")
        for f in files:
            try:
                df_tmp = pd.read_parquet(os.path.join(SAVE_DIR, f), columns=['id', 'gel'])
                for site in SITES.keys():
                    site_df = df_tmp[df_tmp['gel'] == site]
                    if not site_df.empty:
                        min_ids[site] = min(min_ids[site], site_df['id'].min())
            except Exception as e:
                print(f"파일 읽기 오류 ({f}): {e}")
        
        last_file_num = int(files[-1].replace('.parquet', ''))
        next_chunk_num = (last_file_num // CHUNK_SIZE) + 1

    # 사이트별 시작 ID 최종 결정
    start_ids = {}
    for site in SITES.keys():
        if min_ids[site] != float('inf'):
            start_ids[site] = int(min_ids[site]) - 1 # 기존 최소값에서 1을 뺀 값부터 아래로 수집
            print(f"[{site}] 기존 데이터 발견. ID {start_ids[site]:,} 부터 아래로 수집합니다.")
        else:
            start_ids[site] = get_latest_id_from_api(site)
            
    return start_ids

# ───────────────────────────────────────────────
# 데이터 저장 로직
# ───────────────────────────────────────────────
def save_chunk_if_needed():
    global buffer, next_chunk_num, saved_chunks
    with data_lock:
        if len(buffer) >= CHUNK_SIZE:
            chunk_to_save = buffer[:CHUNK_SIZE]
            buffer = buffer[CHUNK_SIZE:]
            
            filename = f'{next_chunk_num * CHUNK_SIZE}.parquet'
            save_path = os.path.join(SAVE_DIR, filename)
            pd.DataFrame(chunk_to_save).to_parquet(save_path, index=False)
            
            saved_chunks += 1
            next_chunk_num += 1

def save_remaining():
    global buffer
    with data_lock:
        if not buffer:
            return
        filename = f'final_mixed_{int(time.time())}.parquet'
        pd.DataFrame(buffer).to_parquet(os.path.join(SAVE_DIR, filename), index=False)
        print(f"\n[완료] 남은 {len(buffer)}건 저장 완료: {filename}")
        buffer = []

# ───────────────────────────────────────────────
# API 요청
# ───────────────────────────────────────────────
def fetch_api_data(site, post_id):
    url = SITES[site]
    params = {'tags': f'id:{post_id}'}
    if site in ['s', 'g']:
        params.update({'page': 'dapi', 's': 'post', 'q': 'index'})
    
    headers = {'User-Agent': ua.random}
    
    try:
        response = session.get(url, params=params, headers=headers, timeout=15)

        if response.status_code in [403, 429]:
            return None, 'denied'
        if response.status_code != 200:
            return None, 'error'

        root = ET.fromstring(response.content)
        post = root.find('post')
        if post is None:
            return None, 'empty'

        post_data = dict(post.attrib)

        for key, value in list(post_data.items()):
            if isinstance(value, str) and value.isdigit():
                post_data[key] = int(value)
            elif key.endswith('_url') and isinstance(value, str) and value.startswith('//'):
                post_data[key] = 'https:' + value

        post_data['gel'] = site
        return post_data, 'success'
    except Exception:
        return None, 'error'

# ───────────────────────────────────────────────
# 큐에 ID를 공급하는 피더(Feeder) 스레드
# ───────────────────────────────────────────────
def queue_feeder(site, start_id, end_id, q):
    for pid in range(start_id, end_id - 1, -1):
        q.put(pid)
    # 워커들에게 종료 신호(None) 전송
    for _ in range(WORKERS_PER_SITE):
        q.put(None)

# ───────────────────────────────────────────────
# 실제 작업을 수행하는 워커(Worker) 스레드
# ───────────────────────────────────────────────
def worker(site, q, pbar):
    while True:
        pause_events[site].wait() # 사이트별 차단 시 개별 정지
        
        post_id = q.get()
        if post_id is None:
            q.task_done()
            break

        result, status = fetch_api_data(site, post_id)

        if status == 'denied':
            with data_lock:
                stats[site]['denied'] += 1
            
            # 해당 사이트만 일시 정지
            if pause_events[site].is_set():
                pause_events[site].clear()
                print(f"\n[!] {site} 차단 감지(ID: {post_id}). 해당 사이트 워커만 {BLOCK_SLEEP}초간 대기합니다...")
                
                def unpause_site():
                    time.sleep(BLOCK_SLEEP)
                    pause_events[site].set()
                threading.Thread(target=unpause_site, daemon=True).start()
            
            q.put(post_id) # 실패한 ID 복귀
            q.task_done()
            continue

        with data_lock:
            stats[site][status] += 1
            if result:
                buffer.append(result)

        save_chunk_if_needed()
        
        # 통합 진행률 표시 업데이트
        pbar.update(1)
        with data_lock:
            total_success = sum(s['success'] for s in stats.values())
            pbar.set_postfix({'성공': total_success, '버퍼': len(buffer), '저장': saved_chunks}, refresh=False)
        
        q.task_done()
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

# ───────────────────────────────────────────────
# 메인 실행
# ───────────────────────────────────────────────
if __name__ == "__main__":
    start_ids = get_resume_info()
    
    # 총 수행해야 할 작업량 계산
    total_tasks = sum((start_ids[site] - END_ID + 1) for site in SITES.keys())
    
    print(f"\n=== 3개 사이트 통합 수집 시작 (총 워커: {len(SITES) * WORKERS_PER_SITE}개) ===")
    pbar = tqdm(total=total_tasks, desc="총 진행률", ncols=100)

    # 1. 큐에 작업을 넣어주는 피더 스레드 3개 시작
    feeder_threads = []
    for site in SITES.keys():
        ft = threading.Thread(target=queue_feeder, args=(site, start_ids[site], END_ID, queues[site]))
        ft.daemon = True
        ft.start()
        feeder_threads.append(ft)

    # 2. 데이터를 수집하는 워커 스레드 9개 시작 (각 사이트당 3개)
    worker_threads = []
    for site in SITES.keys():
        for _ in range(WORKERS_PER_SITE):
            wt = threading.Thread(target=worker, args=(site, queues[site], pbar))
            wt.daemon = True
            wt.start()
            worker_threads.append(wt)

    try:
        # 모든 큐의 작업이 끝날 때까지 대기
        for q in queues.values():
            q.join()
    except KeyboardInterrupt:
        print("\n[!] 사용자 중단됨. 남은 데이터를 안전하게 저장합니다...")

    save_remaining()
    pbar.close()
    
    print("\n=== 최종 수집 결과 ===")
    for site, s in stats.items():
        print(f"[{site}] 성공: {s['success']:,} / 없음: {s['empty']:,} / 차단: {s['denied']:,}")

[s] API 최신 ID 확인: 6,657,186
[g] 최신 ID 조회 실패: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))
[d] 최신 ID 조회 실패: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))

=== 3개 사이트 통합 수집 시작 (총 워커: 9개) ===


총 진행률:   0%|            | 1076/26657186 [01:06<290:26:26, 25.49it/s, 성공=323, 버퍼=323, 저장=0]0it/s]